### Catastrophe Command App\n
\n
Builds deterministic catastrophe scenario tables and deploys the standalone command-center app.

In [ ]:
%pip install --upgrade databricks-sdk "psycopg[binary]"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
import re
import sys

CATALOG = dbutils.widgets.get("CATALOG")
SIMULATOR_SCHEMA = dbutils.widgets.get("SIMULATOR_SCHEMA")
OPS_WAREHOUSE_NAME = dbutils.widgets.get("OPS_WAREHOUSE_NAME")
# City and catastrophe are chosen in the app now, so these are no longer job
# parameters. dbutils.widgets.get() RAISES InputWidgetNotDefined when a param
# is absent (it does not return ""), so each get must be guarded.
try:
    CATASTROPHE_SCENARIO = dbutils.widgets.get("CATASTROPHE_SCENARIO") or "bridge_outage"
except Exception:
    CATASTROPHE_SCENARIO = "bridge_outage"
try:
    CATASTROPHE_SEED = int(dbutils.widgets.get("CATASTROPHE_SEED") or "2026")
except Exception:
    CATASTROPHE_SEED = 2026
try:
    CITY = (dbutils.widgets.get("CITY") or "amsterdam").strip().lower()
except Exception:
    CITY = "amsterdam"
try:
    AI_GATEWAY_ENDPOINT_NAME = dbutils.widgets.get("AI_GATEWAY_ENDPOINT_NAME")
except Exception:
    AI_GATEWAY_ENDPOINT_NAME = ""
# Optional fallback chat model for the in-app agent when no AI Gateway is set.
# Empty => the app uses agent.py's built-in default (databricks-claude-sonnet-4).
try:
    AGENT_LLM_FALLBACK = dbutils.widgets.get("AGENT_LLM_FALLBACK")
except Exception:
    AGENT_LLM_FALLBACK = ""

In [ ]:
sys.path.append('../data/canonical')
sys.path.append('../utils')
from catastrophe_scenarios import (
    scenario_catalog_rows,
    generate_incidents_all_cities,
    all_city_kitchen_rows,
    city_config,
    CITIES,
)
from uc_state import add

if CITY not in CITIES:
    print(f"Unknown CITY '{CITY}', falling back to amsterdam")
    CITY = "amsterdam"
CITY_CFG = city_config(CITY)
print(f"Active demo city: {CITY_CFG['name']} (choke: {CITY_CFG['bridge']['name']})")
print(f"DevConnect tour: {len(CITIES)} cities")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SIMULATOR_SCHEMA}")

spark.createDataFrame(scenario_catalog_rows()).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_scenarios"
)
spark.createDataFrame(
    generate_incidents_all_cities(CATASTROPHE_SCENARIO, CATASTROPHE_SEED)
).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_incidents"
)

# Ghost-kitchen locations for every DevConnect tour city (Amsterdam curated,
# the rest generated around each city centre). The app filters by active city.
spark.createDataFrame(all_city_kitchen_rows(seed=CATASTROPHE_SEED)).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_kitchens"
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_actions (
  action_id STRING,
  action_type STRING,
  order_id STRING,
  incident_id STRING,
  notes STRING,
  created_at_utc TIMESTAMP
) USING DELTA
""")

# ── Governed menu + stock master data (UC multi-table transaction demo) ──
# demos/devconnect-runbooks/5-warehouse-transactions-remove-menu-items.sql. When the supply run can't
# cross the closed bridge a kitchen runs out of an ingredient. "86'ing" the
# affected dishes must update BOTH tables in one atomic commit: zero the stock
# (kitchen_inventory) AND flip the menu (menu_availability). Invariant: a dish
# is available only while its ingredient has stock.
from datetime import datetime, timezone
from catastrophe_scenarios import amsterdam_kitchen_rows, generate_city_kitchens
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType, TimestampType,
)

# Menu → key ingredient. Shared ingredients mean one stock-out 86s several dishes.
_MENU = [
    ("Cheeseburger", "burger_buns"),
    ("Bacon BBQ Burger", "burger_buns"),
    ("Meatball Sub", "beef_patty"),
    ("Caesar Salad", "romaine"),
    ("Vanilla Shake", "milk"),
    ("Ice Cream Sundae", "milk"),
]
_INGREDIENTS = ["burger_buns", "beef_patty", "romaine", "milk"]
# Per-ingredient starting stock (units per kitchen). Vary so the 86 demo
# doesn't show identical before/after numbers for every product.
_INGREDIENT_QTY = {
    "burger_buns": 40,
    "beef_patty": 35,
    "romaine": 25,
    "milk": 12,
}
_now = datetime.now(timezone.utc).replace(microsecond=0)

_inv_rows = []
_menu_rows = []
for _cid in CITIES:
    _kitchens = (
        amsterdam_kitchen_rows() if _cid == "amsterdam"
        else generate_city_kitchens(_cid, seed=CATASTROPHE_SEED)
    )
    for _k in _kitchens:
        _kid = _k["kitchen_id"]
        for _ing in _INGREDIENTS:
            _inv_rows.append((_kid, _cid, _ing, _INGREDIENT_QTY[_ing], _now))
        for _item, _ing in _MENU:
            _menu_rows.append((_kid, _cid, _item, _ing, True, _now))

_inv_schema = StructType([
    StructField("kitchen_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("ingredient", StringType(), False),
    StructField("qty", IntegerType(), False),
    StructField("updated_at", TimestampType(), False),
])
_menu_schema = StructType([
    StructField("kitchen_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("item", StringType(), False),
    StructField("ingredient", StringType(), False),
    StructField("available", BooleanType(), False),
    StructField("updated_at", TimestampType(), False),
])

spark.createDataFrame(_inv_rows, schema=_inv_schema).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.kitchen_inventory"
)
spark.createDataFrame(_menu_rows, schema=_menu_schema).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.menu_availability"
)
print(
    f"Seeded kitchen_inventory ({len(_inv_rows)} rows) and "
    f"menu_availability ({len(_menu_rows)} rows) across {len(CITIES)} cities"
)

# Reference crossings for runbook SQL + active city mirror for warehouse queries.
_crossing_rows = [
    (cid, c.bridge_name, c.alt_name, c.river) for cid, c in CITIES.items()
]
_crossing_schema = StructType([
    StructField("city_id", StringType(), False),
    StructField("bridge_name", StringType(), False),
    StructField("alt_name", StringType(), False),
    StructField("river", StringType(), False),
])
spark.createDataFrame(_crossing_rows, schema=_crossing_schema).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.city_crossings"
)
_active_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("city_id", StringType(), False),
    StructField("updated_at", TimestampType(), False),
])
spark.createDataFrame([(1, CITY, _now)], schema=_active_schema).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SIMULATOR_SCHEMA}.demo_active_city"
)
print(
    f"Seeded city_crossings ({len(_crossing_rows)} rows), demo_active_city ({CITY}). "
    "Live warehouse reads use Lakebase CDF → lakebase.lb_orders_history."
)

print(f"Scenario materialized: {CATASTROPHE_SCENARIO} seed={CATASTROPHE_SEED}")

In [ ]:
import json
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import BadRequest, NotFound
from databricks.sdk.service.apps import App, AppDeployment, AppResource, AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission

w = WorkspaceClient()
existing_wh = [wh for wh in w.warehouses.list() if wh.name == OPS_WAREHOUSE_NAME]
if not existing_wh:
    raise RuntimeError(f"Warehouse '{OPS_WAREHOUSE_NAME}' not found")
warehouse = existing_wh[0]

app_name = re.sub(r"-+", "-", re.sub(r"[^a-z0-9-]", "-", f"catastrophe-command-{CATALOG}".lower())).strip("-")[:30]
source_code_path = os.path.abspath("../apps/catastrophe-command")

# Lakebase Autoscaling: the app persists orders/statuses/refunds/complaints to
# the project's DEFAULT `databricks_postgres` database (shipped with the shared
# project provisioned by stages/lakebase_project.ipynb). This target uses NO
# custom database.
LAKEBASE_PROJECT_ID = re.sub(r"[^a-z0-9-]", "-", f"{CATALOG}-caspers".lower())
LAKEBASE_ENDPOINT_PATH = f"projects/{LAKEBASE_PROJECT_ID}/branches/production/endpoints/primary"
LAKEBASE_DATABASE_NAME = "databricks_postgres"

def _yq(v):
    # Render a value as a YAML single-quoted scalar. Inside single quotes
    # YAML treats everything literally EXCEPT a single quote, which must be
    # doubled. Without this, any value containing an apostrophe (e.g. the
    # catastrophe desc \"Berlagebrug's bascule...\") or the JSON in CITY_CONFIG
    # terminates the scalar early and the Apps platform rejects app.yaml with
    # \"Error reading app.yaml file\".
    return "'" + str(v).replace("'", "''") + "'"

_app_env = [
    ("DATABRICKS_CATALOG", CATALOG),
    ("SIMULATOR_SCHEMA", SIMULATOR_SCHEMA),
    ("DATABRICKS_WAREHOUSE_ID", warehouse.id),
    ("OPS_WAREHOUSE_NAME", OPS_WAREHOUSE_NAME),
    ("LAKEBASE_ENDPOINT_PATH", LAKEBASE_ENDPOINT_PATH),
    ("LAKEBASE_DATABASE_NAME", LAKEBASE_DATABASE_NAME),
    ("CATASTROPHE_SCENARIO", CATASTROPHE_SCENARIO),
    ("CATASTROPHE_SEED", CATASTROPHE_SEED),
    ("CITY", CITY),
    ("CITY_NAME", CITY_CFG["name"]),
    ("CITY_CONFIG", json.dumps(CITY_CFG)),
    ("DEMO_THEATER", "amsterdam"),
    ("AI_GATEWAY_ENDPOINT_NAME", AI_GATEWAY_ENDPOINT_NAME),
    ("AGENT_LLM_FALLBACK", AGENT_LLM_FALLBACK),
]
_env_lines = "\n".join(f"  - name: {k}\n    value: {_yq(v)}" for k, v in _app_env)
app_yaml_contents = f"""command:
  - uvicorn
  - app.main:app
env:
{_env_lines}
"""
with open(os.path.abspath("../apps/catastrophe-command/app.yaml"), "w") as _f:
    _f.write(app_yaml_contents)

app_def = App(
    name=app_name,
    description="Casper's Kitchens catastrophe command center",
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
            ),
        ),
    ],
)

def _compute_state_now():
    a = w.apps.get(app_name)
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

try:
    w.apps.get(app_name)
    # Compute must be ACTIVE or STOPPED to update. Wait out transient
    # STARTING/STOPPING/UPDATING states (e.g. from a prior interrupted run) so
    # the update doesn't fail with BadRequest.
    wait_stable = time.time() + 15 * 60
    while _compute_state_now() in ("STARTING", "STOPPING", "UPDATING"):
        print(f"App compute is {_compute_state_now()}; waiting for it to settle before update...")
        if time.time() > wait_stable:
            print("Timed out waiting for stable compute state; attempting update anyway")
            break
        time.sleep(10)
    w.apps.update(app_name, app_def)
    print(f"Updated app: {app_name}")
except NotFound:
    w.apps.create(app_def)
    print(f"Created app: {app_name}")

def _deploy_once():
    return w.apps.deploy(
        app_name=app_name,
        app_deployment=AppDeployment(source_code_path=source_code_path),
    )

def _ensure_compute_running():
    # Bring the app compute to RUNNING before (re)deploying. Deploy requires
    # RUNNING; start() is ONLY valid from STOPPED. A prior interrupted run can
    # leave compute in STARTING/STOPPING/UPDATING, so wait those out first and
    # call start() only when it has actually settled to STOPPED — otherwise
    # start() raises BadRequest ("compute is in STARTING state ... needs to be
    # STOPPED to start").
    settle_deadline = time.time() + 15 * 60
    while True:
        st = _compute_state_now()
        if st in ("ACTIVE", "RUNNING", "READY"):
            return
        if st in ("STARTING", "STOPPING", "UPDATING"):
            print(f"App compute is {st}; waiting for it to settle before start...")
            if time.time() > settle_deadline:
                print("Timed out waiting for compute to settle; attempting start anyway")
                break
            time.sleep(10)
            continue
        # STOPPED / ERROR / FAILED / unknown -> attempt a start.
        break
    try:
        w.apps.start(app_name)
    except BadRequest as se:
        # Raced with another actor that already started it; tolerate and wait.
        print(f"start() rejected ({se}); waiting for compute to reach RUNNING")
    run_deadline = time.time() + 20 * 60
    while True:
        st = _compute_state_now()
        if st in ("ACTIVE", "RUNNING", "READY"):
            return
        if st in ("ERROR", "FAILED"):
            raise RuntimeError(f"App compute entered {st} while starting {app_name}")
        if time.time() > run_deadline:
            raise TimeoutError(f"App compute not RUNNING for {app_name}")
        print(f"App compute is {st}; waiting for RUNNING...")
        time.sleep(10)

def _deploy_state(d):
    st = getattr(d, "status", None)
    s = getattr(st, "state", None) if st is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

def _deploy_and_wait():
    # Kick off one deployment (starting the app first if it isn't RUNNING) and
    # poll it to a terminal state. Returns the final state string.
    try:
        deployment = _deploy_once()
    except BadRequest as e:
        if "not in RUNNING state" not in str(e):
            raise
        print(f"App {app_name} not RUNNING; ensuring compute is up before retrying deploy...")
        _ensure_compute_running()
        deployment = _deploy_once()
    deadline = time.time() + 30 * 60
    while True:
        current_dep = w.apps.get_deployment(app_name=app_name, deployment_id=deployment.deployment_id)
        state = _deploy_state(current_dep)
        print(f"Deployment state: {state}")
        if state in ("SUCCEEDED", "FAILED", "STOPPED"):
            return state
        if time.time() > deadline:
            raise TimeoutError(f"Deployment timeout for {app_name}")
        time.sleep(10)

# App deployments occasionally fail transiently on first boot; retry a couple
# of times before giving up.
_deploy_attempts = 3
for _attempt in range(1, _deploy_attempts + 1):
    _final_state = _deploy_and_wait()
    if _final_state == "SUCCEEDED":
        break
    if _attempt < _deploy_attempts:
        print(f"Deployment attempt {_attempt} ended in {_final_state}; retrying in 20s...")
        time.sleep(20)
    else:
        raise RuntimeError(
            f"Deployment failed for {app_name} after {_deploy_attempts} attempts: state={_final_state}"
        )

def _compute_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

# NOTE: deliberately NO restart here. The new env (CITY / CITY_CONFIG /
# scenario) in app.yaml AND the Lakebase DB + SP role are all applied by the
# SINGLE stop→(recreate DB + grant role)→start cycle in the Lakebase cell below.
# Restarting here too would cold-start the app once WITHOUT a database and then
# again WITH it — two wasted restarts. We only need the app object now to read
# its service principal so we can grant it DB + UC access BEFORE that one start.
app_obj = w.apps.get(app_name)
add(CATALOG, "apps", app_obj)
print(f"App url: {getattr(app_obj, 'url', 'n/a')}")

app_sp_id = (
    getattr(app_obj, "service_principal_client_id", None)
    or (app_obj.as_dict() if hasattr(app_obj, "as_dict") else {}).get("service_principal_client_id")
    or (app_obj.as_dict() if hasattr(app_obj, "as_dict") else {}).get("id")
)
assert app_sp_id, "Could not determine app service principal ID"
print(f"App SP ID: {app_sp_id}")

In [ ]:
from databricks.sdk.service import catalog as catalog_svc

for full_name, securable_type, privilege in [
    (CATALOG, "CATALOG", catalog_svc.Privilege.USE_CATALOG),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.locations", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_scenarios", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_incidents", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_kitchens", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_actions", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_actions", "TABLE", catalog_svc.Privilege.MODIFY),
    # Tables read by the ai UC functions (revenue_at_risk /
    # compare_orders_today_vs_baseline / ingredient_availability / check_menu_consistency). UC functions
    # run with the INVOKER's rights (the app SP), so the SP needs SELECT on every
    # table they touch. The 86 procedure is SQL SECURITY DEFINER, so it needs NO
    # MODIFY grant here. Live order state comes from Lakebase CDF
    # (lakebase.lb_orders_history), same as runbook Q3/Q4.
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.demo_active_city", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakebase", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.lakebase.lb_orders_history", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.kitchen_inventory", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.{SIMULATOR_SCHEMA}.menu_availability", "TABLE", catalog_svc.Privilege.SELECT),
    # Historical analytics tables (built by stages/catastrophe_history.ipynb).
    (f"{CATALOG}.orders", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.orders.bronze_hist_orders", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.orders.bronze_hist_order_events", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.orders.bronze_hist_refunds", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.orders.bronze_hist_complaints", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.orders.bronze_hist_actions", "TABLE", catalog_svc.Privilege.SELECT),
]:
    try:
        w.grants.update(
            full_name=full_name,
            securable_type=securable_type,
            changes=[
                catalog_svc.PermissionsChange(
                    add=[privilege],
                    principal=app_sp_id,
                )
            ],
        )
        print(f"Granted {privilege} on {full_name}")
    except Exception as e:
        print(f"Could not grant {privilege} on {full_name}: {e}")

### UC functions (deployed after Lakebase CDF)

Warehouse actions that read `lakebase.lb_orders_history` are created after the
app is up, Postgres tables exist, and CDF is enabled into `{CATALOG}.lakebase`.


In [ ]:
# ── Persist to Lakebase ──────────────────────────────────────────────────────
# The app persists orders / statuses / refunds / complaints / actions to the
# project's DEFAULT `databricks_postgres` database (shared Autoscaling project
# from stages/lakebase_project.ipynb). This target does NOT create a custom
# database. We grant the app service principal DATABRICKS_SUPERUSER on the branch
# (applies to every DB in the project, including databricks_postgres) so it can
# create and write its own tables on startup, then restart the app once so it
# boots with the LAKEBASE_* env AND the SP role in place and init_db() creates
# the schema. Best-effort: a Lakebase hiccup must not brick the app — it degrades
# gracefully (DB features disabled) if this doesn't land.
import sys as _sys, time as _t
_sys.path.append("../utils")

if app_sp_id:
    try:
        from databricks.sdk.service.postgres import Role, RoleRoleSpec, RoleMembershipRole, RoleIdentityType
        from databricks.sdk.common.types.fieldmask import FieldMask
        _branch = f"projects/{LAKEBASE_PROJECT_ID}/branches/production"
        _roles = list(w.postgres.list_roles(_branch))
        _role = next((r for r in _roles if getattr(r.spec, "postgres_role", None) == app_sp_id), None)
        if _role:
            _role.spec = RoleRoleSpec(
                identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                postgres_role=app_sp_id,
                membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
            )
            w.postgres.update_role(
                name=_role.name, role=_role,
                update_mask=FieldMask(field_mask=["spec.membership_roles"]),
            )
            print("Updated app SP role to DATABRICKS_SUPERUSER")
        else:
            try:
                w.postgres.create_role(
                    parent=_branch,
                    role=Role(spec=RoleRoleSpec(
                        identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                        postgres_role=app_sp_id,
                        membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
                    )),
                )
                print("Created DATABRICKS_SUPERUSER role for app SP")
            except Exception as _ce:
                if "already exists" in str(_ce).lower():
                    print("App SP role already exists — skipping")
                else:
                    raise
    except Exception as e:
        print(f"Could not grant Lakebase superuser to app SP: {e}")

# Restart so the app boots with the DB env + role present (init_db creates tables).
def _cs(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""
try:
    w.apps.stop(app_name)
    _dl = _t.time() + 10 * 60
    while _cs(w.apps.get(app_name)) not in ("STOPPED", "", None):
        if _t.time() > _dl:
            break
        _t.sleep(5)
except Exception as e:
    print(f"Stop skipped: {e}")
try:
    w.apps.start(app_name)
except Exception as e:
    print(f"Start failed: {e}")
_dl = _t.time() + 30 * 60
while True:
    _st = _cs(w.apps.get(app_name))
    print(f"App state: {_st}")
    if _st in ("ACTIVE", "RUNNING", "READY"):
        break
    if _st in ("ERROR", "FAILED"):
        raise RuntimeError(f"App entered failure state: {_st}")
    if _t.time() > _dl:
        raise TimeoutError("App not ready after restart")
    _t.sleep(15)
print("Lakebase wired; app restarted.")

In [ ]:
# ── Wait for app Postgres tables + REPLICA IDENTITY FULL ──────────────────────
# App owns the tables (creates them on boot in db.py). Poll until `orders`
# exists and every public table is REPLICA IDENTITY FULL. CDF comes next.
import time as _t2
import psycopg

_pg_host = w.postgres.get_endpoint(name=LAKEBASE_ENDPOINT_PATH).status.hosts.host
_pg_user = w.current_user.me().user_name
_pg_db = LAKEBASE_DATABASE_NAME

def _pg_conn():
    _tok = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT_PATH).token
    _c = psycopg.connect(
        host=_pg_host, port=5432, dbname=_pg_db, user=_pg_user,
        password=_tok, sslmode="require", connect_timeout=30,
    )
    _c.autocommit = True
    return _c

_lbl = {"f": "FULL", "d": "DEFAULT", "n": "NOTHING", "i": "INDEX"}
_status = []
_deadline = _t2.time() + 8 * 60
while _t2.time() < _deadline:
    try:
        with _pg_conn() as _c, _c.cursor() as _cur:
            _cur.execute(
                "SELECT table_name FROM information_schema.tables "
                "WHERE table_schema='public' AND table_type='BASE TABLE'"
            )
            for (_t,) in _cur.fetchall():
                try:
                    _cur.execute(f'ALTER TABLE public."{_t}" REPLICA IDENTITY FULL')
                except Exception:
                    pass
            _cur.execute(
                "SELECT c.relname, c.relreplident FROM pg_class c "
                "JOIN pg_namespace n ON n.oid = c.relnamespace "
                "WHERE n.nspname='public' AND c.relkind='r' ORDER BY c.relname"
            )
            _status = _cur.fetchall()
    except Exception as _e:
        print(f"Waiting for Lakebase: {_e}")
        _status = []
    if any(n == "orders" for n, _ in _status) and all(ri == "f" for _, ri in _status):
        break
    print("Waiting for app tables + REPLICA IDENTITY FULL; retrying in 10s...")
    _t2.sleep(10)

if not any(n == "orders" for n, _ in _status):
    raise RuntimeError("App has not created public.orders yet — cannot enable CDF")
if any(ri != "f" for _, ri in _status):
    raise RuntimeError(
        "Not all public tables are REPLICA IDENTITY FULL: "
        + ", ".join(f"{n}={_lbl.get(ri, ri)}" for n, ri in _status)
    )
print(f"Ready for CDF ({len(_status)} public tables, all FULL):")
for _name, _ri in _status:
    print(f"  - {_name}: {_lbl.get(_ri, _ri)}")


In [ ]:
# ── Enable Lakebase CDF (after app tables exist) ──────────────────────────────
# public → {CATALOG}.lakebase. Keep an existing same-dest config (do not
# delete+recreate every deploy — that creates lb_*_history_1/_2 orphans).
# Seed one orders row first — CDF skips empty tables.
import sys as _sys_cdf
_sys_cdf.path.append("../utils")
from lakebase_cdf import enable_cdf, wait_for_cdf_table

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.lakebase")

with _pg_conn() as _c, _c.cursor() as _cur:
    _cur.execute(
        """
        INSERT INTO public.orders (
            order_id, session_id, city, kitchen, vehicle, kind,
            cold, status, late_min
        ) VALUES (
            '__cdf_bootstrap__', 'bootstrap', 'NYC', 'bootstrap', 'bike', 'hot',
            false, 'placed', 0
        )
        ON CONFLICT (order_id) DO NOTHING
        """
    )
    _cur.execute("SELECT count(*) FROM public.orders")
    print(f"orders rows: {_cur.fetchone()[0]}")

_cdf = enable_cdf(
    w,
    LAKEBASE_PROJECT_ID,
    catalog=CATALOG,
    schema="lakebase",
    postgres_schema="public",
)
print(_cdf)

wait_for_cdf_table(spark, f"{CATALOG}.lakebase.lb_orders_history", timeout_s=20 * 60)

from databricks.sdk.service import catalog as _cat_cdf
try:
    w.grants.update(
        full_name=f"{CATALOG}.lakebase",
        securable_type="SCHEMA",
        changes=[_cat_cdf.PermissionsChange(
            add=[_cat_cdf.Privilege.USE_SCHEMA], principal=app_sp_id
        )],
    )
    w.grants.update(
        full_name=f"{CATALOG}.lakebase.lb_orders_history",
        securable_type="TABLE",
        changes=[_cat_cdf.PermissionsChange(
            add=[_cat_cdf.Privilege.SELECT], principal=app_sp_id
        )],
    )
    print(f"Granted app SP SELECT on {CATALOG}.lakebase.lb_orders_history")
except Exception as _ge:
    print(f"Grant on CDF table (best-effort): {_ge}")


In [ ]:
# ── Vetted warehouse actions as UC functions + a procedure (Act 2) ───────────
# The in-app catastrophe agent (apps/catastrophe-command/app/agent.py) embeds NO
# SQL. Its `warehouse` actions call these deployed objects by name
# (`SELECT * FROM {CATALOG}.ai.<fn>()` / `CALL ...remove_ingredient_from_menu...()`),
# so the query bodies ship + version with the bundle. Bodies mirror Q3/Q4/Q5a/
# 5a/5/5b in demos/devconnect-runbooks/5-warehouse-transactions-remove-menu-items.sql.
#
# Read actions are UC SQL table functions (RETURNS TABLE, schema inferred). The
# 86 action is a UC PROCEDURE — UC functions cannot do DML, so the two-table
# write must be a procedure. It is SQL SECURITY DEFINER (runs as its creator, the
# deployer, who owns the tables) so the app SP needs only EXECUTE, not MODIFY.
# Created on the ops warehouse via the Statement Execution API (procedures /
# SQL scripting are a DBSQL feature). The ai schema lives under
# {CATALOG}; `cleanup` drops the catalog and takes these with it.
import time as _t_uc

_UC_SCHEMA = f"{CATALOG}.ai"
_ST = f"{CATALOG}.{SIMULATOR_SCHEMA}"
_ORDERS = f"{CATALOG}.orders"
_LB = f"{CATALOG}.lakebase"


def _wh_run(stmt: str, label: str):
    r = w.statement_execution.execute_statement(
        warehouse_id=warehouse.id, statement=stmt, wait_timeout="30s"
    )
    sid = r.statement_id
    while r.status and r.status.state and r.status.state.value in ("PENDING", "RUNNING"):
        _t_uc.sleep(1)
        r = w.statement_execution.get_statement(sid)
    state = r.status.state.value if r.status and r.status.state else "UNKNOWN"
    if state != "SUCCEEDED":
        err = getattr(r.status, "error", None)
        raise RuntimeError(f"[{label}] {state}: {err}\n--- SQL ---\n{stmt[:600]}")
    print(f"[{label}] OK")
    return r


_wh_run(f"CREATE SCHEMA IF NOT EXISTS {_UC_SCHEMA}", "schema ai")

_FUNCTIONS = {
    "revenue_at_risk": f"""
CREATE OR REPLACE FUNCTION {_UC_SCHEMA}.revenue_at_risk()
RETURNS TABLE
READS SQL DATA
COMMENT 'Revenue at risk right now by status + item, valued from this city 90-day avg order value. Mirrors 3-warehouse-estimate-revenue-at-risk.sql (Lakebase CDF lb_orders_history).'
RETURN
WITH active AS (
    SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1
),
live_orders AS (
    SELECT order_id, session_id, city, status, kind, late_min, updated_at
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _sort_by DESC) AS rn
        FROM {_LB}.lb_orders_history
    ) o
    WHERE rn = 1
      AND _pg_change_type IN ('insert', 'update_postimage')
),
latest AS (
    -- Prefer the active demo city; if no live orders there (stale demo_active_city),
    -- fall back to whichever city has the most recent order.
    SELECT session_id, city FROM (
        SELECT o.session_id, o.city,
               ROW_NUMBER() OVER (
                   ORDER BY CASE WHEN o.city = a.city_id THEN 0 ELSE 1 END, o.updated_at DESC
               ) AS rn
        FROM live_orders o
        CROSS JOIN active a
    ) WHERE rn = 1
),
hist_val AS (
    SELECT city_id, kind, ROUND(AVG(order_value), 2) AS avg_order_value
    FROM {_ORDERS}.bronze_hist_orders
    GROUP BY city_id, kind
),
live_open AS (
    SELECT o.order_id, o.city, o.status, o.kind AS kind_label,
        CASE o.kind
            WHEN 'Hot food'  THEN 'hot'
            WHEN 'Groceries' THEN 'grocery'
            WHEN 'Ice cream' THEN 'ice'
            WHEN 'Frozen'    THEN 'frozen'
        END AS kind_code
    FROM live_orders o
    WHERE o.session_id = (SELECT session_id FROM latest)
      AND o.status IN ('placed', 'routing', 'enroute', 'stuck', 'late', 'rerouted')
)
SELECT
    lo.status,
    lo.kind_label AS item,
    COUNT(*) AS orders_at_risk,
    ROUND(SUM(COALESCE(h.avg_order_value, 20.00)), 2) AS est_revenue_at_risk
FROM live_open lo
LEFT JOIN hist_val h ON h.city_id = lo.city AND h.kind = lo.kind_code
GROUP BY ROLLUP (lo.status, lo.kind_label)
ORDER BY lo.status NULLS LAST, est_revenue_at_risk DESC
""",
    "compare_orders_today_vs_baseline": f"""
CREATE OR REPLACE FUNCTION {_UC_SCHEMA}.compare_orders_today_vs_baseline()
RETURNS TABLE
READS SQL DATA
COMMENT 'This run vs a normal day in this city: orders, cancel %, disrupted %, avg lateness. Mirrors 4-warehouse-compare-today-vs-normal.sql (Lakebase CDF lb_orders_history).'
RETURN
WITH active AS (
    SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1
),
live_orders AS (
    SELECT order_id, session_id, city, status, kind, late_min, updated_at
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _sort_by DESC) AS rn
        FROM {_LB}.lb_orders_history
    ) o
    WHERE rn = 1
      AND _pg_change_type IN ('insert', 'update_postimage')
),
latest AS (
    -- Prefer the active demo city; if no live orders there (stale demo_active_city),
    -- fall back to whichever city has the most recent order.
    SELECT session_id, city FROM (
        SELECT o.session_id, o.city,
               ROW_NUMBER() OVER (
                   ORDER BY CASE WHEN o.city = a.city_id THEN 0 ELSE 1 END, o.updated_at DESC
               ) AS rn
        FROM live_orders o
        CROSS JOIN active a
    ) WHERE rn = 1
),
live AS (
    SELECT
        COUNT(*) AS orders,
        ROUND(100.0 * AVG(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END), 1) AS cancel_pct,
        ROUND(100.0 * AVG(CASE WHEN status IN ('stuck', 'rerouted') THEN 1 ELSE 0 END), 1) AS disrupted_pct,
        ROUND(AVG(late_min), 1) AS avg_late_min
    FROM live_orders
    WHERE session_id = (SELECT session_id FROM latest)
),
hist AS (
    SELECT
        ROUND(100.0 * AVG(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END), 1) AS cancel_pct,
        ROUND(100.0 * AVG(CASE WHEN disrupted THEN 1 ELSE 0 END), 1) AS disrupted_pct,
        ROUND(AVG(late_min), 1) AS avg_late_min
    FROM {_ORDERS}.bronze_hist_orders h
    JOIN latest l ON h.city_id = l.city
)
SELECT 'today (this run)' AS period, l.orders, l.cancel_pct, l.disrupted_pct, l.avg_late_min FROM live l
UNION ALL
SELECT 'normal (90-day avg)' AS period, CAST(NULL AS BIGINT), h.cancel_pct, h.disrupted_pct, h.avg_late_min FROM hist h
""",
    "ingredient_availability": f"""
CREATE OR REPLACE FUNCTION {_UC_SCHEMA}.ingredient_availability(p_ingredient STRING)
RETURNS TABLE
READS SQL DATA
COMMENT 'Stock qty + how many dishes are still available for the given ingredient, active city. Run BEFORE 86ing. Generalizes 5-warehouse-transactions-remove-menu-items.sql 5a to any ingredient (param named p_ingredient to avoid clashing with the ingredient column).'
RETURN
WITH active AS (
    SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1
)
SELECT 'stock' AS what, SUM(i.qty) AS stock_qty, CAST(NULL AS BIGINT) AS available_dishes
FROM   {_ST}.kitchen_inventory i
JOIN   active a ON i.city_id = a.city_id
WHERE  i.ingredient = p_ingredient
UNION ALL
SELECT 'menu', CAST(NULL AS BIGINT), COUNT(*)
FROM   {_ST}.menu_availability m
JOIN   active a ON m.city_id = a.city_id
WHERE  m.ingredient = p_ingredient AND m.available = true
""",
    "check_menu_consistency": f"""
CREATE OR REPLACE FUNCTION {_UC_SCHEMA}.check_menu_consistency(p_ingredient STRING)
RETURNS TABLE
READS SQL DATA
COMMENT 'After 86ing the given ingredient: stock, dishes still available, and inconsistent rows (available dish w/ zero stock, MUST be 0). Generalizes 5-warehouse-transactions-remove-menu-items.sql 5b to any ingredient.'
RETURN
WITH active AS (
    SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1
)
SELECT
  (SELECT COALESCE(SUM(i.qty), 0)
     FROM {_ST}.kitchen_inventory i
     JOIN active a ON i.city_id = a.city_id
     WHERE i.ingredient = p_ingredient) AS stock_qty,
  (SELECT COUNT(*)
     FROM {_ST}.menu_availability m
     JOIN active a ON m.city_id = a.city_id
     WHERE m.ingredient = p_ingredient AND m.available = true) AS still_available,
  (SELECT COUNT(*)
     FROM {_ST}.menu_availability m
     JOIN {_ST}.kitchen_inventory i
       ON i.kitchen_id = m.kitchen_id AND i.ingredient = m.ingredient
     JOIN active a ON m.city_id = a.city_id
     WHERE m.available = true AND i.qty = 0 AND m.ingredient = p_ingredient) AS inconsistent_rows
""",
}

_PROCEDURE = f"""
CREATE OR REPLACE PROCEDURE {_UC_SCHEMA}.remove_ingredient_from_menu(p_ingredient STRING)
LANGUAGE SQL
SQL SECURITY DEFINER
MODIFIES SQL DATA
COMMENT '86 the given ingredient: zero its stock AND flip its menu dishes unavailable for the active city. Generalizes 5-warehouse-transactions-remove-menu-items.sql atomic block to any ingredient.'
AS BEGIN
  UPDATE {_ST}.kitchen_inventory
  SET    qty = 0, updated_at = current_timestamp()
  WHERE  city_id = (SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1)
    AND  ingredient = p_ingredient;

  UPDATE {_ST}.menu_availability
  SET    available = false, updated_at = current_timestamp()
  WHERE  city_id = (SELECT city_id FROM {_ST}.demo_active_city WHERE id = 1)
    AND  ingredient = p_ingredient;
END
"""

# Drop previous names so renames don't leave orphaned UC objects.
for _old in (
    "today_vs_normal",
    "ingredient_stock_before",
    "ingredient_after_check",
):
    _wh_run(f"DROP FUNCTION IF EXISTS {_UC_SCHEMA}.{_old}", f"drop old fn {_old}")
_wh_run(
    f"DROP PROCEDURE IF EXISTS {_UC_SCHEMA}.eightysix_ingredient",
    "drop old procedure eightysix_ingredient",
)

for _fn_name, _fn_sql in _FUNCTIONS.items():
    _wh_run(_fn_sql, f"function {_fn_name}")
_wh_run(_PROCEDURE, "procedure remove_ingredient_from_menu")

# Grant the app SP USE SCHEMA + EXECUTE. GRANT via SQL (works for both FUNCTION
# and PROCEDURE securables; the SDK grants API's securable types don't cleanly
# cover procedures). Best-effort per grant.
_grants = [
    f"GRANT USE SCHEMA ON SCHEMA {_UC_SCHEMA} TO `{app_sp_id}`",
    *[f"GRANT EXECUTE ON FUNCTION {_UC_SCHEMA}.{_fn} TO `{app_sp_id}`" for _fn in _FUNCTIONS],
    f"GRANT EXECUTE ON PROCEDURE {_UC_SCHEMA}.remove_ingredient_from_menu TO `{app_sp_id}`",
]
for _g in _grants:
    try:
        _wh_run(_g, "grant")
    except Exception as _ge:
        print(f"Could not apply grant: {_ge}")

print(f"ai: {len(_FUNCTIONS)} functions + 1 procedure deployed and granted to app SP")

# ── Grant app SP access to the Unity AI Gateway model service ────────────────
# The agent (apps/catastrophe-command/app/agent.py, GATEWAY_ENDPOINT) routes its
# tool-calling LLM calls through this Unity AI Gateway *model service* — a UC
# securable (securable_type "model_service"). The app runs as its service
# principal, which needs USE CATALOG + USE SCHEMA on the service's parent
# catalog/schema and EXECUTE on the service itself, else the gateway returns
# 404 NOT_FOUND for the SP (definer's rights: the SP does NOT need access to the
# underlying model). SQL `GRANT ... ON SERVICE` and the SDK grants API don't
# cover this securable yet, so use the UC permissions REST API. Best-effort so
# deploys where the endpoint is absent don't fail the stage.
_MODEL_SERVICE = f"{CATALOG}.default.command-agent"  # keep in sync w/ agent.py _default_gateway_endpoint()
_ms_catalog, _ms_schema, _ = _MODEL_SERVICE.split(".")
try:
    for _sec_name, _sec_type, _sec_priv in [
        (_ms_catalog, "CATALOG", catalog_svc.Privilege.USE_CATALOG),
        (f"{_ms_catalog}.{_ms_schema}", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    ]:
        w.grants.update(
            full_name=_sec_name,
            securable_type=_sec_type,
            changes=[catalog_svc.PermissionsChange(add=[_sec_priv], principal=app_sp_id)],
        )
    w.api_client.do(
        "PATCH",
        f"/api/2.1/unity-catalog/permissions/model_service/{_MODEL_SERVICE}",
        body={"changes": [{"principal": app_sp_id, "add": ["EXECUTE"]}]},
    )
    print(f"Granted app SP EXECUTE on model service {_MODEL_SERVICE}")
except Exception as _e:
    print(f"Could not grant model-service access ({_MODEL_SERVICE}); "
          f"if the agent 404s, grant EXECUTE on it manually: {_e}")